# **Preparation of SMILES Training Data for Generative Models**

This notebook prepares molecular training data for the generative AI model. The cleaned activity dataset for a selected target is filtered to retain compounds with pIC50 > 5. RDKit is then used to validate, sanitize, and convert SMILES to canonical form, while removing invalid entries, salts (keeping the largest fragment), and duplicate molecules.

The final dataset of unique, chemically valid SMILES is saved both as a validated CSV file and as a .smi file, which serves as the input for training the generative model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
# Load cleaned dataset

targets = ["5-HT6","ache", "bace1", "buche","esr1","3beta", "mao-b"]

target=targets[6]
df = pd.read_csv(f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/processed-data/activity-labeled-data/{target}_cleaned_labeled_smiles.csv")

In [ ]:
df.head()

,molecule_chembl_id,canonical_smiles,IC50_nM,pIC50,target_chembl_id,bioactivity_class
0,CHEMBL350093,N#CCCN1CC(=O)OC(c2ccc(OCc3ccccc3)cc2)=N1,18.0,7.744727,CHEMBL2039,active
1,CHEMBL161907,O=c1c(=O)c2ccc(OCCCC(F)(F)F)cc2c1=O,9.0,8.045757,CHEMBL2039,active
2,CHEMBL17079,N#CCCn1nc(-c2ccc(OCc3ccccc3)cc2)oc1=S,4.4,8.356547,CHEMBL2039,active
3,CHEMBL160347,COc1cc(Br)c2oc(C3CCNCC3)cc2c1,23400.0,4.630784,CHEMBL2039,inactive
4,CHEMBL347197,CCN=C1CCc2c1n(C)c1ccccc21,77600.0,4.110138,CHEMBL2039,inactive


In [ ]:
# Filter for pIC50 > 5
df_filtered = df[df['pIC50'] > 5]

# Save to new file
df_filtered.to_csv(f'/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data/{target}_filtered_pIC50_gt5.csv', index=False)

print(f"Filtered dataset saved with {len(df_filtered)} entries.")

Filtered dataset saved with 3744 entries.


# **Extract SMILES for training**

In [ ]:
df_filtered['canonical_smiles'].to_csv(f'/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data/{target}_smiles_for_training.smi', index=False, header=False)


# **Clean , de-dupe SMILES**

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.1/36.1 MB 51.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from pathlib import Path
from rdkit import Chem

targets = ["5-HT6","ache", "bace1", "buche","esr1","3beta", "mao-b"]
target = targets[6]

# ---- paths ----
src_csv = Path(f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/processed-data/activity-labeled-data/{target}_cleaned_labeled_smiles.csv")
out_dir = Path("/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv_valid = out_dir / f"{target}_filtered_valid_unique.csv"
out_smi      = out_dir / f"{target}_smiles_for_training.smi"

# ---- load & potency filter ----
df = pd.read_csv(src_csv)
df = df[df["pIC50"] > 5].copy()

# pick the SMILES column
for cand in ["canonical_smiles", "SMILES", "smiles"]:
    if cand in df.columns:
        smi_col = cand
        break
else:
    raise KeyError(f"No SMILES column found in {src_csv}. Expected: canonical_smiles / SMILES / smiles")

# ---- RDKit parse + canonicalize + keep largest fragment ----
def sanitize_to_canonical(s: str, keep_largest_fragment: bool = True) -> str | None:
    s = (s or "").strip()
    if not s:
        return None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None
    if keep_largest_fragment and "." in s:
        # keep only the largest fragment by atom count
        frags = Chem.GetMolFrags(m, asMols=True, sanitize=False)
        if not frags:
            return None
        m = max(frags, key=lambda x: x.GetNumAtoms())
        Chem.SanitizeMol(m)
    # canonical, isomeric SMILES (keeps stereochem if present)
    return Chem.MolToSmiles(m, canonical=True)

src_n = len(df)
df["canonical_smiles"] = df[smi_col].astype(str).map(sanitize_to_canonical)

parsable_n = df["canonical_smiles"].notna().sum()
df = df.dropna(subset=["canonical_smiles"]).copy()

# ---- de-duplicate on canonical ----
df = df.drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)
uniq_n = len(df)

# ---- save validated CSV + training SMI ----
df.to_csv(out_csv_valid, index=False)
df["canonical_smiles"].to_csv(out_smi, index=False, header=False)

print(f"[Validity] Start (pIC50>5): {src_n}")
print(f"[Validity] RDKit-parsable:  {parsable_n}")
print(f"[Validity] Unique canonical: {uniq_n}")
print(f"[Out] CSV: {out_csv_valid}")
print(f"[Out] SMI: {out_smi}")


[Validity] Start (pIC50>5): 3744
[Validity] RDKit-parsable:  3744
[Validity] Unique canonical: 2957
[Out] CSV: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data/mao-b_filtered_valid_unique.csv
[Out] SMI: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data/mao-b_smiles_for_training.smi
